In [1]:
import numpy as np
import scipy.io
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, Flatten, TimeDistributed, Input
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
battery_names = ['B0005', 'B0006', 'B0007', 'B0018']

def extract_voltage_and_soh(mat_file, battery_name):
    data = scipy.io.loadmat(mat_file)
    cycles = data[battery_name]['cycle'][0][0][0]
    
    voltages = []
    capacities = []
    
    for i in range(cycles.shape[0]):
        cycle = cycles[i]
        if cycle['type'][0] == 'discharge':
            voltage = cycle['data'][0]['Voltage_measured'][0][0].flatten()
            current = cycle['data'][0]['Current_measured'][0][0].flatten()
            time = cycle['data'][0]['Time'][0][0].flatten()
            capacity = np.trapezoid(abs(current), time) / 3600
            voltages.append(voltage)
            capacities.append(capacity)
    
    initial = capacities[0]
    soh = np.array([cap / initial * 100 for cap in capacities])
    return voltages, soh

voltages_all = {}
soh_all = {}

for name in battery_names:
    v, s = extract_voltage_and_soh(f'../data/raw/{name}.mat', name)
    voltages_all[name] = v
    soh_all[name] = s
    print(f"{name}: {len(v)} discharge cycles, voltage lengths: min={min(len(x) for x in v)}, max={max(len(x) for x in v)}")

B0005: 168 discharge cycles, voltage lengths: min=179, max=371
B0006: 168 discharge cycles, voltage lengths: min=179, max=371
B0007: 168 discharge cycles, voltage lengths: min=179, max=371
B0018: 132 discharge cycles, voltage lengths: min=200, max=366


In [3]:
SEQUENCE_LENGTH = 200

def standardise_voltages(voltages, length=SEQUENCE_LENGTH):
    standardised = []
    for v in voltages:
        if len(v) >= length:
            standardised.append(v[:length])
        else:
            padded = np.pad(v, (0, length - len(v)), mode='edge')
            standardised.append(padded)
    return np.array(standardised)

voltages_std = {}
for name in battery_names:
    voltages_std[name] = standardise_voltages(voltages_all[name])
    print(f"{name}: {voltages_std[name].shape}")

B0005: (168, 200)
B0006: (168, 200)
B0007: (168, 200)
B0018: (132, 200)


In [4]:
def create_cnn_sequences(voltages, soh, look_back=10):
    X, y = [], []
    for i in range(len(soh) - look_back):
        X.append(voltages[i : i + look_back])
        y.append(soh[i + look_back])
    return np.array(X), np.array(y)

sequences = {}
for name in battery_names:
    X, y = create_cnn_sequences(voltages_std[name], soh_all[name])
    sequences[name] = (X, y)
    print(f"{name} — X shape: {X.shape}, y shape: {y.shape}")

X_train = np.concatenate([sequences['B0005'][0], sequences['B0006'][0], sequences['B0007'][0]])
y_train = np.concatenate([sequences['B0005'][1], sequences['B0006'][1], sequences['B0007'][1]])
X_test = sequences['B0018'][0]
y_test = sequences['B0018'][1]

print(f"\nX_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")

B0005 — X shape: (158, 10, 200), y shape: (158,)
B0006 — X shape: (158, 10, 200), y shape: (158,)
B0007 — X shape: (158, 10, 200), y shape: (158,)
B0018 — X shape: (122, 10, 200), y shape: (122,)

X_train: (474, 10, 200)
X_test: (122, 10, 200)


In [5]:
X_train_scaled = X_train / 4.2
X_test_scaled = X_test / 4.2

y_scaler = MinMaxScaler()
y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1))
y_test_scaled = y_scaler.transform(y_test.reshape(-1, 1))

print(f"X_train range: {X_train_scaled.min():.2f} to {X_train_scaled.max():.2f}")
print(f"y_train range: {y_train_scaled.min():.2f} to {y_train_scaled.max():.2f}")

X_train range: 0.41 to 1.01
y_train range: 0.00 to 1.00


In [6]:
X_train_final = X_train_scaled.reshape(X_train_scaled.shape[0], 10, 200, 1)
X_test_final = X_test_scaled.reshape(X_test_scaled.shape[0], 10, 200, 1)

print(f"X_train_final: {X_train_final.shape}")
print(f"X_test_final: {X_test_final.shape}")

X_train_final: (474, 10, 200, 1)
X_test_final: (122, 10, 200, 1)


In [7]:
def extract_features(voltages):
    features = []
    for v in voltages:
        mean_v = np.mean(v)
        min_v = np.min(v)
        voltage_drop = v[0] - v[-1]
        area = np.trapezoid(v) / len(v)
        slope = np.polyfit(np.arange(len(v)), v, 1)[0]
        std_v = np.std(v)
        mid = len(v) // 2
        knee_v = v[mid]
        discharge_time = len(v)
        features.append([mean_v, min_v, voltage_drop, area, slope, std_v, knee_v, discharge_time])
    return np.array(features)

features_all = {}
for name in battery_names:
    features_all[name] = extract_features(voltages_std[name])
    print(f"{name}: {features_all[name].shape}")

B0005: (168, 8)
B0006: (168, 8)
B0007: (168, 8)
B0018: (132, 8)


In [8]:
def create_cycle_split_sequences(features, soh, look_back=10, train_ratio=0.8):
    X, y = [], []
    for i in range(len(soh) - look_back):
        X.append(features[i : i + look_back])
        y.append(soh[i + look_back])
    X, y = np.array(X), np.array(y)
    split = int(len(X) * train_ratio)
    return X[:split], y[:split], X[split:], y[split:]

X_train_list, y_train_list = [], []
X_test_list, y_test_list = [], []

for name in battery_names:
    X_tr, y_tr, X_te, y_te = create_cycle_split_sequences(features_all[name], soh_all[name])
    X_train_list.append(X_tr)
    y_train_list.append(y_tr)
    X_test_list.append(X_te)
    y_test_list.append(y_te)

X_train = np.concatenate(X_train_list)
y_train = np.concatenate(y_train_list)
X_test = np.concatenate(X_test_list)
y_test = np.concatenate(y_test_list)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")

X_train: (475, 10, 8)
X_test: (121, 10, 8)


In [9]:
n_samples_train, n_timesteps, n_features = X_train.shape

X_train_reshaped = X_train.reshape(-1, n_features)
X_test_reshaped = X_test.reshape(-1, n_features)

X_scaler = MinMaxScaler()
X_train_scaled = X_scaler.fit_transform(X_train_reshaped).reshape(X_train.shape)
X_test_scaled = X_scaler.transform(X_test_reshaped).reshape(X_test.shape)

y_scaler = MinMaxScaler()
y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1))
y_test_scaled = y_scaler.transform(y_test.reshape(-1, 1))

print(f"X_train_scaled: {X_train_scaled.shape}")
print(f"X_test_scaled: {X_test_scaled.shape}")

X_train_scaled: (475, 10, 8)
X_test_scaled: (121, 10, 8)


In [10]:
model_v2 = Sequential([
    Input(shape=(10, 8)),
    LSTM(32, return_sequences=False),
    Dropout(0.3),
    Dense(8, activation='relu'),
    Dense(1)
])

model_v2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history_v2 = model_v2.fit(
    X_train_scaled, y_train_scaled,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    shuffle=False,
    verbose=1
)

Epoch 1/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 47ms/step - loss: 0.5063 - val_loss: 0.0182
Epoch 2/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.1042 - val_loss: 0.0098
Epoch 3/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0794 - val_loss: 0.0206
Epoch 4/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0622 - val_loss: 0.0219
Epoch 5/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0514 - val_loss: 0.0294
Epoch 6/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0506 - val_loss: 0.0400
Epoch 7/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0466 - val_loss: 0.0366
Epoch 8/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.0381 - val_loss: 0.0349
Epoch 9/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.0370 - val_loss: 0.0427
Epoch 10/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.0349 - val_loss: 0.0482
Epoch 11/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.0347 - val_loss: 0.0385
Epoch 12/100
24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step